## Creation of Catalog and Schema

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ecommerce_lakehouse;
CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.raw;
CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.bronze;
CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.silver;
CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.gold; 

## Importing Neccesary Packages

In [0]:
from common.schema import *
from pyspark.sql import functions as F

## Function for Ingesting the data

In [0]:

def ingest_data(volume_path,file_name,schema,catalog = 'ecommerce_lakehouse',
schema_name = 'bronze'):
    
    source_path = f"{volume_path}/{file_name}.csv"
    table_name = f"{catalog}.{schema_name}.{file_name.replace('olist_', '').replace('_dataset', '')}"

    df = (spark.read.format('csv')
        .option('header',True)
        .schema(schema)
        .load(source_path)
        .withColumn('_load_timestamp',F.current_timestamp())
        .withColumn('_file_name',F.col('_metadata.file_name')))

    (df.write.format('delta').mode('overwrite')
        .saveAsTable(table_name))
    
    return table_name

## Ingesting the data

In [0]:
files_and_schemas = {
    'olist_customers_dataset' : customer_schema,
    'olist_geolocation_dataset' : geolocation_schema,
    'olist_order_items_dataset' : order_items_schema,
    'olist_order_payments_dataset' : order_payments_schema,
    'olist_order_reviews_dataset' : order_reviews_schema,
    'olist_orders_dataset' : orders_schema,
    'olist_products_dataset' : products_schema,
    'olist_sellers_dataset' : sellers_schema,
    'product_category_name_translation' : product_category_name_translation_schema
}

catalog = 'ecommerce_lakehouse'
schema_name = 'bronze'
volume_path = '/Volumes/ecommerce_lakehouse/raw/data'

for file_name,schema in files_and_schemas.items():
    t = ingest_data(volume_path,file_name,schema,catalog,schema_name)
    print(f"Ingested {t} to Bronze layer")

## Validation

In [0]:
for file_name in files_and_schemas.keys():
    t = f"{catalog}.{schema_name}.{file_name.replace("olist_",'').replace("_dataset",'')}"
    cnt = spark.table(t).count()

    print(f"{t} has {cnt} records")